# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)  
- Croissant `@id`: `https://api.app.sen.science/frontiers/7154140/8875020f-242a-4f80-afd4-a4cbac8715f0`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all available record sets in the dataset with their `@id` and field `@id`s. This will help select the appropriate components for further analysis.


In [ ]:
# List available record sets and their fields

def list_record_sets(ds):
    rsets = getattr(ds.metadata, 'record_sets', [])
    if not rsets:
        # Try alternate attribute
        rsets = getattr(ds.metadata, 'recordSet', [])
    print(f"Found {len(rsets)} record sets in the dataset:\n")
    for recset in rsets:
        print(f"- Record set: {recset['@id'] if isinstance(recset, dict) else recset}")
        # Try to extract fields
        if isinstance(recset, dict) and 'fields' in recset:
            for field in recset['fields']:
                print(f"    - Field: {field['@id']} ({field.get('name', '')})")
        elif isinstance(recset, dict) and 'field' in recset:
            for field in recset['field']:
                if isinstance(field, dict):
                    print(f"    - Field: {field['@id']} ({field.get('name', '')})")
                else:
                    print(f"    - Field: {field}")
        else:
            print("    (Fields info not available. See documentation or use dataset.schema for details)")

list_record_sets(dataset)

# If the dataset does not have explicit record_sets, mlcroissant usually exposes at least one default record set (with full data matrix). If needed, we can inspect the schema JSON directly.
# We'll also print all available @id references in the schema for manual inspection if the above is empty.

if not getattr(dataset.metadata, 'recordSet', []) and not getattr(dataset.metadata, 'record_sets', []):
    print("\nNo record sets found under the usual keys. Listing all @id fields found in dataset schema for manual inspection.\n")
    with open(dataset.local_schema_path, encoding='utf-8') as f:
        schema_json = json.load(f)
    def walk(obj):
        if isinstance(obj, dict):
            if '@id' in obj:
                print(obj['@id'])
            for v in obj.values():
                walk(v)
        elif isinstance(obj, list):
            for v in obj:
                walk(v)
    walk(schema_json)
    print("(See above for all @id entries. Use the relevant @id for record_set in the next step.)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Here we will try to extract all data tables by ID.
# Replace the record_set_ids list below after inspecting the Overview output above or by referring to documentation.

# For this FAIR2 dataset, retrieve the correct record set @ids from the schema:

import re

# Helper to find all "cr:RecordSet" entities
with open(dataset.local_schema_path, encoding='utf-8') as f:
    schema = json.load(f)

def find_record_sets(schema):
    record_sets = []
    def walk(o):
        if isinstance(o, dict):
            if '@type' in o:
                for t in o['@type'] if isinstance(o['@type'], list) else [o['@type']]:
                    if re.search(r'(RecordSet)$', t):
                        record_sets.append(o)
            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for v in o:
                walk(v)
    walk(schema)
    return record_sets

record_sets_found = find_record_sets(schema)
record_set_ids = [rs.get('@id') for rs in record_sets_found if '@id' in rs]

print(f"Found Record Sets: {record_set_ids}\n")
# Use the first record set for further analysis
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())

# If there are no record sets, we'll attempt to load with a guessed default, which is usually the main table.
if not dataframes:
    print("No explicit record_set found, attempting default read...")
    try:
        df = pd.DataFrame(dataset.records())
        dataframes['default'] = df
        print(df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Unable to infer data: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming distributions, or grouping data by key attributes. All field/column references must use their `@id`.

*Below, we showcase filtering, normalization, and grouping. **You must adjust the `numeric_field` and `group_field` variable values to match the actual column `@id`s from step 3.***

In [ ]:
# Configure the record set and numeric field; replace with those discovered above
if dataframes:
    # Take the first record set as an example
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]
    print(f"Using record set: {example_record_set_id}")
    
    # Let's infer a numeric field
    # We'll pick the first float/integer typed column
    # Optionally, inspect dtypes
    print("Column datatypes:")
    print(df.dtypes)
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    print(f"Numeric columns found: {numeric_cols}")
    if numeric_cols:
        numeric_field = numeric_cols[0]  # Example
    else:
        # Fallback: try to guess a numeric field name
        match = [c for c in df.columns if 'count' in c.lower() or 'number' in c.lower()]
        numeric_field = match[0] if match else df.columns[0]
    print(f"Using numeric field: {numeric_field}")

    # Try to use the first string/object column as category
    group_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field = group_cols[0] if group_cols else None
    print(f"Using group field: {group_field}")

    # Filtering
    threshold = df[numeric_field].quantile(0.75)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

    # Grouping
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]
    # Plot histogram for selected numeric column
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No DataFrame to visualize.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load the FAIR² dataset using the Croissant schema and `mlcroissant`
- Discover and enumerate available record sets and fields (referenced by their `@id`)
- Extract a record set into a DataFrame for inspection and further analysis
- Perform basic exploratory data analysis: filtering, normalization, and grouping using field `@id`s
- Visualize field distributions and relationships

This approach is fully reproducible and consistently references dataset content via Croissant `@id`, facilitating robust, maintainable research workflows.

**For further analysis, consult the dataset schema documentation to map additional field and record set `@id`s to your analysis questions.**